In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_2.py ──
"""
Shared infrastructure for MLFP03 Exercise 2 — Regularisation and
Cross-Validation.

Contains: data loading for the Singapore credit scoring dataset,
feature preparation, synthetic 1D bias-variance problem, alpha grids,
plotting helpers, and a shared OUTPUT_DIR for generated artefacts.

Technique-specific code (Ridge/Lasso model construction, nested CV
loops, learning curves) lives in the per-technique solution files —
this module holds only the helpers those files share.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

# Deterministic seed for every random operation in this exercise
SEED = 42

# Alpha sweep used by Ridge / Lasso / ElasticNet and the regularisation path
ALPHAS: list[float] = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# Output directory for HTML plots, summary CSVs, etc.
OUTPUT_DIR = Path("outputs") / "mlfp03_ex2_regularisation_cv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> (
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]
):
    """Load + preprocess the MLFP02 Singapore credit scoring parquet.

    Returns:
        X_train, y_train, X_test, y_test, feature_names

    Target: predicts ``credit_utilization`` (continuous ratio). Falls
    back to ``income_sgd`` if the utilisation column is absent in older
    dataset revisions.

    Uses ``kailash_ml.PreprocessingPipeline`` for normalisation +
    ordinal encoding + median imputation. All regularised models
    REQUIRE normalised features (otherwise the penalty is unevenly
    distributed across the coefficient vector).
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    target_col = (
        "credit_utilization" if "credit_utilization" in credit.columns else "income_sgd"
    )

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=credit,
        target=target_col,
        train_size=0.8,
        seed=SEED,
        normalize=True,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != target_col]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=target_col,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=target_col,
    )

    return X_train, y_train, X_test, y_test, col_info["feature_columns"]


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 1D PROBLEM — for bias/variance and polynomial fits
# ════════════════════════════════════════════════════════════════════════


def make_sine_dataset(
    n: int = 100,
    noise_sigma: float = 0.2,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Generate a 1D noisy-sine regression problem.

    Returns:
        x_train_2d (n_train, 1), y_train (n_train,),
        x_test_2d (n_test, 1),   y_test  (n_test,),
        noise_variance (float, σ² — the irreducible error floor)

    The true function is ``y = sin(2πx) + ε`` with ε ~ N(0, σ²).
    The noise variance is returned so callers can use it in the
    bias-variance decomposition (σ² is the "irreducible noise" term).
    """
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    y = np.sin(2 * np.pi * x) + rng.normal(0, noise_sigma, n)

    split = int(n * 0.8)
    x_train = x[:split].reshape(-1, 1)
    x_test = x[split:].reshape(-1, 1)
    y_train = y[:split]
    y_test = y[split:]
    return x_train, y_train, x_test, y_test, noise_sigma**2


def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial-features + scaler + linear-regression pipeline."""
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LinearRegression()),
        ]
    )


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_header(title: str) -> None:
    """Print a banner so each phase of a technique file is easy to spot."""
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def format_results_table(rows: list[dict[str, Any]], cols: list[str]) -> str:
    """Render a small table of dicts as fixed-width text."""
    header = "  ".join(f"{c:>12}" for c in cols)
    sep = "-" * len(header)
    body = "\n".join(
        "  ".join(
            f"{row[c]:>12.4f}" if isinstance(row[c], float) else f"{row[c]:>12}"
            for c in cols
        )
        for row in rows
    )
    return f"{header}\n{sep}\n{body}"


def save_html_plot(fig: Any, name: str) -> Path:
    """Write a plotly figure into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / name
    fig.write_html(str(path))
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 2.5: Learning Curves and Diagnostic Playbook
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Read a learning curve to decide "more data" vs "better model"
#   - Compare OLS, Ridge, and Lasso learning curves on one dataset
#   - Recognise the three canonical shapes: converged-far, converged-close,
#     and not-yet-converged
#   - Use learning curves to justify (or reject) data-collection spend
#   - Tie the entire exercise together with a Singapore decision playbook
#
# PREREQUISITES:
#   - 01 through 04 in this exercise
#
# ESTIMATED TIME: ~35 minutes
#
# TASKS (5-phase R10):
#   1. Theory — three learning-curve shapes and what they mean
#   2. Build — three models to compare (OLS, Ridge, Lasso)
#   3. Train — sklearn.learning_curve for each
#   4. Visualise — HTML plots for each model's train-vs-test curve
#   5. Apply — StarHub churn-scoring data-acquisition decision
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import learning_curve

from kailash_ml import ModelVisualizer



## THEORY — Reading a Learning Curve


In [ ]:
# A learning curve plots model performance (train and validation) as a
# function of the training-set size. Three canonical shapes exist:
#
#   1. CONVERGED-FAR APART  — train much higher than test, both flat
#      Diagnosis: HIGH BIAS. More data WON'T help; you need a richer
#      model or more features. This is classic underfitting.
#
#   2. CONVERGED-CLOSE       — train and test curves meet at a good score
#      Diagnosis: You're done. Additional data gives marginal returns.
#
#   3. NOT-YET-CONVERGED    — test still rising as training size grows
#      Diagnosis: HIGH VARIANCE. More data WILL help. Invest in data
#      collection OR regularise harder.
#
# The gap between train and test curves is the VARIANCE component; the
# absolute level of the train curve upper-bounds the achievable bias.



## TASK 2 — BUILD the comparison models


In [ ]:
print_header("Learning Curves — OLS vs Ridge vs Lasso")
X_train, y_train, X_test, y_test, feature_names = load_credit_data()
print(f"Train: {X_train.shape}")

models = {
    "OLS (unregularised)": LinearRegression(),
    "Ridge (α=1)": Ridge(alpha=1.0),
    "Lasso (α=0.1)": Lasso(alpha=0.1, max_iter=10_000),
}

train_sizes_frac = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]



## TASK 3 — TRAIN each model across growing training-set fractions


In [ ]:
all_curves: dict[str, dict[str, np.ndarray]] = {}
for name, model in models.items():
    train_sizes, tr_scores, te_scores = learning_curve(
        model,
        X_train,
        y_train,
        train_sizes=train_sizes_frac,
        cv=5,
        scoring="r2",
        n_jobs=-1,
    )
    all_curves[name] = {
        "sizes": train_sizes,
        "train_mean": tr_scores.mean(axis=1),
        "test_mean": te_scores.mean(axis=1),
        "train_std": tr_scores.std(axis=1),
        "test_std": te_scores.std(axis=1),
    }

    print_header(name)
    print(f"{'N':>8} {'Train R²':>10} {'Test R²':>10} {'Gap':>10}")
    print("-" * 40)
    for n_, tr, te in zip(train_sizes, tr_scores.mean(axis=1), te_scores.mean(axis=1)):
        print(f"{n_:>8} {tr:>10.4f} {te:>10.4f} {(tr - te):>10.4f}")



### Checkpoint 1


In [ ]:
assert len(train_sizes) == len(
    train_sizes_frac
), "Should have one entry per training-size fraction"
assert all(
    "test_mean" in c for c in all_curves.values()
), "Every model should record a test-mean curve"
print("\n[ok] Checkpoint 1 passed — learning curves computed for all models")



## TASK 4 — VISUALISE the three learning curves


In [ ]:
# Save one HTML per model: train vs test R² as a function of training-
# set size. These plots are the single best diagnostic you can show a
# credit committee when asking for a data-collection budget.

viz = ModelVisualizer()
saved_paths: list[str] = []
for name, curves in all_curves.items():
    fig = viz.training_history(
        {
            f"{name} — train": curves["train_mean"].tolist(),
            f"{name} — test": curves["test_mean"].tolist(),
        },
        x_label="Training set size (samples)",
    )
    fig.update_layout(title=f"Learning Curve — {name}")
    safe = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    safe = safe.replace("=", "_").replace("α", "a")
    path = save_html_plot(fig, f"learning_curve_{safe}.html")
    saved_paths.append(path.name)

print("\nSaved learning-curve plots:")
for p in saved_paths:
    print(f"  {p}")



### Checkpoint 2


In [ ]:
assert len(saved_paths) == len(models), "One plot per model should be saved"
print("\n[ok] Checkpoint 2 passed — all learning-curve plots written")
# INTERPRETATION:
#   - If OLS's test curve is still RISING at full size, more data will
#     improve OLS. Regularised models should converge earlier.
#   - If Ridge's test curve is FLAT and the gap is tiny, Ridge has
#     reached the ceiling; more data won't help, but a richer model
#     (kernel, tree) might.
#   - If Lasso's test curve is above Ridge at small N, Lasso's feature
#     selection is helping when data is scarce.



## TASK 5 — APPLY: StarHub Mobile Churn Scoring Data Decision


Learning curve shape       | Decision                       | S$ impact
---------------------------|--------------------------------|-----------
Converged-close (flat)     | PASS on extra data; buy        |  +S$1.4M
                           | features instead               |  saved
---------------------------|--------------------------------|-----------
Not-yet-converged (rising) | BUY the extra data             |  +S$1.39M/yr
                           |                                |  recurring
---------------------------|--------------------------------|-----------
Converged-far (high bias)  | PASS on data; investigate      |  +S$1.4M
                           | non-linear models or           |  saved +
                           | cross-product features         |  redirected


In [ ]:
# SCENARIO: StarHub (Singapore telco, ~2.2M mobile subscribers) is
# deciding whether to buy an additional 18 months of historical CDR
# (call-detail-record) data from a partner carrier. The data would
# expand the churn-model training set from ~180K labelled subscribers
# to ~420K, at a cost of S$1.4M for the licence + integration.
#
# WHY LEARNING CURVES ARE THE RIGHT TOOL:
#   - The CTO needs a defensible answer: "will another S$1.4M of data
#     actually reduce churn?" A learning curve gives a quantitative
#     yes/no BEFORE the cheque is written.
#   - If the current churn model's test curve is FLAT (converged-close),
#     the S$1.4M won't pay back — the model has already extracted all
#     the signal its feature set can express. Spend the money on
#     BETTER features instead.
#   - If the curve is STILL RISING (not-yet-converged), the incremental
#     data will lift test R² by an estimable amount, which translates
#     directly into retained-revenue projections.
#
# CALCULATION (illustrative numbers):
#   - Current monthly churn: 1.4% × 2.2M subscribers × S$42 ARPU
#     = S$1.29M/month of recurring revenue lost to churn.
#   - Learning curve extrapolation shows adding 240K labelled rows lifts
#     test AUC by +0.028, which the retention team translates into a
#     relative churn reduction of ~9% in the top-decile risk cohort.
#   - Revenue saved: 9% × S$1.29M × 12 months = ~S$1.39M/year.
#   - Payback: S$1.4M data cost / S$1.39M annual savings ≈ 12 months.
#     Green-light the purchase.
#
# COUNTERFACTUAL: Without the learning curve, the data science team
# would either (a) over-claim and buy the data even if the model was
# already converged, wasting S$1.4M, or (b) under-claim and pass on
# the data, foregoing the S$1.39M/year saving. Either way, the wrong
# decision costs ~S$1.4M+. The learning curve is the S$0 diagnostic
# that anchors the ~S$1.4M decision.

print_header("StarHub Churn Scoring — Learning Curve Data Decision")
print(
)



## REFLECTION — The Full Exercise in One Frame


======================================================================
  WHAT YOU'VE MASTERED (ENTIRE EXERCISE)
======================================================================

  01. Bias-Variance     — why "more complex" is not always "better"
  02. Ridge (L2)        — smooth shrinkage, Gaussian prior, stability
  03. Lasso + ElasticNet — sparsity, L1 diamond, feature selection
  04. Cross-validation  — nested / time-series / group, match deployment
  05. Learning curves   — diagnose data hunger vs model weakness

  DECISION PLAYBOOK FOR A SINGAPORE ML TEAM:
    Step 1. Start with a learning curve on a Ridge baseline.
    Step 2. If the curve is still rising, invest in more data AND use
            nested CV to pick α honestly.
    Step 3. If the curve is flat but the gap is large, switch to a
            richer model class (not just more data).
    Step 4. If the data has temporal or group structure, DO NOT use
            shuffled k-fold — use TimeSeriesSplit or GroupKFold.
    Step 5. For governance-critical work, prefer Lasso/ElasticNet so
            feature selection is auditable.

  KEY INSIGHT: Regularisation is how you encode your prior belief about
  the world. Cross-validation is how you audit that belief against
  reality. Learning curves tell you whether reality will change if you
  throw more data at it.

  NEXT: Exercise 3 — full supervised model zoo (SVM, KNN, Naive Bayes,
  Trees, Random Forests) on Singapore e-commerce churn data.


In [ ]:
print(
)

